In [1]:
# ==========================================
# BÀI 8: XỬ LÝ DỮ LIỆU THIẾU (ĐỘC LẬP)
# ==========================================
import pandas as pd
import numpy as np
import re
from google.colab import drive

# 1. Mount Drive và Tải dữ liệu
drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/Nhap_Mon_Hoc_May/Buoi2/Du_lieu_thuc_hanh_chuan_bi_du_lieu.xlsx'

df_raw = pd.read_excel(file_path, sheet_name='Du_lieu_tho')
df_clean = df_raw.copy()

# 2. Tái hiện các bước tiền xử lý kiểu dữ liệu (Bài 6 & 7) để có dữ liệu chuẩn cho Bài 8
def clean_price(val):
    if pd.isna(val): return np.nan
    val_str = str(val).lower().strip()
    if 'miễn phí' in val_str or 'free' in val_str: return 0.0
    val_str = re.sub(r'[đ|vnd|\s]', '', val_str).replace('.', '')
    try:
        p = float(val_str)
        return p if p >= 0 else np.nan
    except: return np.nan

def clean_rating(val):
    if pd.isna(val): return np.nan
    m = re.search(r'\d+(\.\d+)?', str(val).replace(',', '.'))
    if m:
        r = float(m.group())
        return r if 1.0 <= r <= 5.0 else np.nan
    return np.nan

df_clean['Price_VND'] = df_clean['Price_VND'].apply(clean_price)
df_clean['Rating'] = df_clean['Rating'].apply(clean_rating)
df_clean['Created_Date'] = pd.to_datetime(df_clean['Created_Date'], errors='coerce')
df_clean.drop_duplicates(inplace=True)

print("=== BẮT ĐẦU THỰC HIỆN BÀI 8 ===\n")


# ------------------------------------------
# Yêu cầu 34: Tạo bảng tỷ lệ thiếu sau khi chuyển kiểu
# ------------------------------------------
print("--- 34. BẢNG TỶ LỆ DỮ LIỆU THIẾU SAU KHI CHUYỂN KIỂU ---")
missing_after_convert = pd.DataFrame({
    'Số lượng thiếu': df_clean.isnull().sum(),
    'Tỷ lệ thiếu (%)': (df_clean.isnull().mean() * 100).round(2)
}).sort_values(by='Số lượng thiếu', ascending=False)

display(missing_after_convert)
print("\n" + "="*60 + "\n")


# ------------------------------------------
# Yêu cầu 35: Price_VND - Điền median theo Category
# ------------------------------------------
df_clean['Price_VND'] = df_clean.groupby('Category')['Price_VND'].transform(
    lambda x: x.fillna(x.median() if not x.median() != x.median() else df_clean['Price_VND'].median())
)
print("-> 35. Đã điền Price_VND bằng trung vị (Median) theo từng danh mục Category.")


# ------------------------------------------
# Yêu cầu 36: Rating - Điền median theo Category và tạo Rating_Missing_Flag
# ------------------------------------------
df_clean['Rating_Missing_Flag'] = df_clean['Rating'].isnull()

df_clean['Rating'] = df_clean.groupby('Category')['Rating'].transform(
    lambda x: x.fillna(x.median() if not x.median() != x.median() else df_clean['Rating'].median())
)
print("-> 36. Đã tạo cờ Rating_Missing_Flag và điền Rating bằng median theo Category.")


# ------------------------------------------
# Yêu cầu 37: Seller_City - Dùng nhãn 'Unknown' kèm giải thích
# ------------------------------------------
df_clean['Seller_City'] = df_clean['Seller_City'].fillna('Unknown')
print("-> 37. Đã gán nhãn 'Unknown' cho vị trí thiếu ở cột Seller_City.")


# ------------------------------------------
# Yêu cầu 38: Description - Điền "Không có mô tả" và tạo Description_Missing_Flag
# ------------------------------------------
df_clean['Description_Missing_Flag'] = df_clean['Description'].isnull()
df_clean['Description'] = df_clean['Description'].fillna("Không có mô tả")
print("-> 38. Đã điền 'Không có mô tả' và tạo cờ Description_Missing_Flag.")


# ------------------------------------------
# Yêu cầu 39: Created_Date - Loại bỏ các dòng bị NaT (Tỷ lệ thiếu rất nhỏ)
# ------------------------------------------
before_date_drop = len(df_clean)
df_clean = df_clean.dropna(subset=['Created_Date']).copy()
after_date_drop = len(df_clean)

print(f"-> 39. Đã loại bỏ {before_date_drop - after_date_drop} dòng có Created_Date bị NaT.")


# ------------------------------------------
# Giải thích lựa chọn xử lý & Báo cáo kết quả
# ------------------------------------------
print("\n[GIẢI THÍCH LỰA CHỌN THUẬT TOÁN]:")
print("- Seller_City (Yêu cầu 37): Sử dụng nhãn 'Unknown' thay vì Mode (yếu vị) vì địa điểm người bán mang tính định danh địa lý, việc tự ý áp đặt thành phố phổ biến nhất có thể gây sai lệch mô hình phân tích vùng miền.")
print("- Created_Date (Yêu cầu 39): Do tỷ lệ ngày lỗi/thiếu cực kỳ nhỏ và không được tự bịa mốc thời gian ngẫu nhiên, giải pháp tối ưu là loại bỏ trực tiếp các dòng NaT này để giữ tính trung thực theo chuỗi thời gian.")

print("\n--- KIỂM TRA LẠI TỶ LỆ THIẾU SAU BÀI 8 ---")
print(f"Tổng số bản ghi sạch hoàn chỉnh: {len(df_clean)}")
print(f"Số lượng giá trị khuyết còn lại: {df_clean.isnull().sum().sum()}")

Mounted at /content/drive
=== BẮT ĐẦU THỰC HIỆN BÀI 8 ===

--- 34. BẢNG TỶ LỆ DỮ LIỆU THIẾU SAU KHI CHUYỂN KIỂU ---


,Số lượng thiếu,Tỷ lệ thiếu (%)
Description,15,6.73
Rating,14,6.28
Price_VND,11,4.93
Created_Date,8,3.59
Seller_City,7,3.14
Product_Name,0,0.00
Purchase_Count,0,0.00
Category,0,0.00
Product_ID,0,0.00
Stock,0,0.00




-> 35. Đã điền Price_VND bằng trung vị (Median) theo từng danh mục Category.
-> 36. Đã tạo cờ Rating_Missing_Flag và điền Rating bằng median theo Category.
-> 37. Đã gán nhãn 'Unknown' cho vị trí thiếu ở cột Seller_City.
-> 38. Đã điền 'Không có mô tả' và tạo cờ Description_Missing_Flag.
-> 39. Đã loại bỏ 8 dòng có Created_Date bị NaT.

[GIẢI THÍCH LỰA CHỌN THUẬT TOÁN]:
- Seller_City (Yêu cầu 37): Sử dụng nhãn 'Unknown' thay vì Mode (yếu vị) vì địa điểm người bán mang tính định danh địa lý, việc tự ý áp đặt thành phố phổ biến nhất có thể gây sai lệch mô hình phân tích vùng miền.
- Created_Date (Yêu cầu 39): Do tỷ lệ ngày lỗi/thiếu cực kỳ nhỏ và không được tự bịa mốc thời gian ngẫu nhiên, giải pháp tối ưu là loại bỏ trực tiếp các dòng NaT này để giữ tính trung thực theo chuỗi thời gian.

--- KIỂM TRA LẠI TỶ LỆ THIẾU SAU BÀI 8 ---
Tổng số bản ghi sạch hoàn chỉnh: 215
Số lượng giá trị khuyết còn lại: 0
